# 02 · Базовая линия

Замер устроен одинаково для всех методов и живёт в `src.report.evaluate`: жадная генерация на обоих
тестах, автопроверки, судья, perplexity и preference accuracy на dev, запись `runs/<имя>-<тест>.json`.
Судья — та же базовая модель с выключенным адаптером: слабее человека, но одинаков для всех.

| метрика | что считает | лучше |
|---|---|---|
| judge | доля ответов с вердиктом PASS по рубрике | выше |
| refusal_judge | то же только на ситуациях с обязательным отказом | выше |
| checks_all | доля ситуаций, где выполнены все назначенные автопроверки | выше |
| false_refusal | доля обычных запросов, где ответ начался с отказа | ниже |
| perplexity | $\exp$ средней по токенам $-\log \pi_\theta$ эталонов dev | ниже |
| pref_acc | доля пар dev, где эталон вероятнее плохого ответа на токен | выше |

Доли идут с интервалом Уилсона, на 100 ситуациях около ±10 пунктов, на 33 около ±16.
Сдвиг меньше — не результат.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer, metrics, report

model, tokenizer = infer.load_model()
print(infer.memory())

In [ ]:
results = report.evaluate(model, tokenizer, "base", note="base model, no training")
report.show()

## Судья против людей

В тесте продукта у части ситуаций есть вердикт человека для ответа продового агента. Прогоняем
те же ответы через нашего судью: это единственная калибровка, которая у нас есть.

In [ ]:
product = list(data.load("test_product"))
labelled = [r for r in product if r["reference"]["human"] in ("PASS", "SOFT-PASS", "FAIL", "SOFT-FAIL")]
verdicts = infer.judge(model, tokenizer, labelled, [r["reference"]["answer"] for r in labelled])
human = [r["reference"]["human"].endswith("PASS") for r in labelled]
agree = sum(a == b for a, b in zip(verdicts, human))
print(f"размечено {len(labelled)}, согласие судьи с человеком {agree}/{len(labelled)} = {agree / len(labelled):.0%}")
print(f"судья засчитал продовому агенту {sum(verdicts)}/{len(labelled)}, люди {sum(human)}/{len(labelled)}")

## Ответы, не только числа

In [ ]:
for i in (0, 10, 28):
    detail = results["product"]["rows"][i]
    data.show(product[i], results["product"]["answers"][i])
    print("проверки:", detail["checks"], "| судья:", "PASS" if detail["judge"] else "FAIL")
    print()